# Tutorial 18b: APIM Monitoring & Log Analytics

## Querying APIM Logs for Compliance & Debugging

### What You'll Learn

Query Azure Monitor logs to:
- See all MCP tool calls through APIM
- Track authentication successes and failures
- Measure latency for SLA compliance
- Identify usage patterns and anomalies

**Prerequisites:**
- Completed Tutorial 18 (APIM MCP registration)
- APIM diagnostic settings enabled
- Log Analytics workspace configured
- `Log Analytics Reader` role on your workspace

**Duration:** 15 minutes

## Part 1: Configuration

Load environment variables and configure the Log Analytics workspace.

In [ ]:
# Configuration - Load environment variables
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# APIM Configuration (from Tutorial 18)
APIM_NAME = os.getenv("APIM_NAME", "")
APIM_GATEWAY_URL = os.getenv("APIM_GATEWAY_URL", "")

# Log Analytics Configuration
LOG_ANALYTICS_WORKSPACE_ID = os.getenv("LOG_ANALYTICS_WORKSPACE_ID", "")

print("=" * 70)
print("Tutorial 18b: APIM Monitoring & Log Analytics")
print("=" * 70)
print(f"""
  APIM Instance:
    Name:           {APIM_NAME}
    Gateway URL:    {APIM_GATEWAY_URL}
    
  Log Analytics:
    Workspace ID:   {LOG_ANALYTICS_WORKSPACE_ID[:20] + '...' if LOG_ANALYTICS_WORKSPACE_ID else '❌ Not configured'}
""")

if not LOG_ANALYTICS_WORKSPACE_ID:
    print("⚠️  To use this notebook, add LOG_ANALYTICS_WORKSPACE_ID to your .env file")
    print("   Find it in: Azure Portal → Log Analytics workspace → Properties → Workspace ID")
else:
    print("✅ Ready to query logs")
print("=" * 70)

## Part 2: Enable APIM Diagnostic Settings

### 🏗️ Architect's Perspective

> *"Before we can query logs, APIM needs to send them somewhere. Diagnostic settings connect APIM to Log Analytics."*

### Check/Enable in Azure Portal

1. **Navigate**: Azure Portal → API Management → your APIM instance
2. **Open**: Monitoring → Diagnostic settings
3. **Create** (if not exists): Click **+ Add diagnostic setting**
4. **Configure**:
   | Setting | Value |
   |---------|-------|
   | Name | `apim-to-log-analytics` |
   | Logs | ✅ GatewayLogs, ✅ WebSocketConnectionLogs |
   | Destination | ✅ Send to Log Analytics workspace |
   | Workspace | Select your workspace |

5. **Save**: Click Save

### 💡 Note on Log Latency

Logs typically take **2-5 minutes** to appear in Log Analytics after requests are made.

In [ ]:
# Initialize Log Analytics client
from azure.monitor.query import LogsQueryClient
from azure.identity import DefaultAzureCredential
from datetime import timedelta

print("=" * 70)
print("Initializing Azure Monitor Query Client")
print("=" * 70)

try:
    credential = DefaultAzureCredential()
    logs_client = LogsQueryClient(credential)
    print("\n✅ Log Analytics client initialized")
    print("   Ready to query APIM diagnostic logs")
except Exception as e:
    print(f"\n❌ Failed to initialize client: {e}")
    print("   Ensure you have 'Log Analytics Reader' role on the workspace")

## Part 3: Query APIM Gateway Logs

### 🏗️ Architect's Perspective

> *"The AzureDiagnostics table captures every request through APIM. This is your audit trail."*

### What the Gateway Logs Tell Us

| Column | Meaning | Audit Value |
|--------|---------|-------------|
| `TimeGenerated` | When the request occurred | Timeline |
| `httpMethod_s` | GET, POST, etc. | Operation type |
| `url_s` | Full request URL | What was accessed |
| `httpStatus_d` | Response code (200, 401, etc.) | Success/failure |
| `apimSubscriptionId_s` | Which subscription key | Who accessed |
| `timeTaken_d` | Request duration (ms) | SLA tracking |
| `clientIP_s` | Caller's IP address | Origin tracking |

In [ ]:
# Query 1: APIM Gateway Logs - All MCP Requests
print("=" * 70)
print("📡 APIM Gateway Logs (AzureDiagnostics)")
print("=" * 70)

if not LOG_ANALYTICS_WORKSPACE_ID:
    print("\n⚠️  LOG_ANALYTICS_WORKSPACE_ID not configured - skipping query")
else:
    try:
        # Query for MCP-related requests in the last 30 minutes
        apim_query = """
        AzureDiagnostics
        | where ResourceProvider == "MICROSOFT.APIMANAGEMENT"
        | where Category == "GatewayLogs"
        | where TimeGenerated > ago(30m)
        | where url_s contains "mcp" or apiId_s contains "mcp"
        | project 
            TimeGenerated,
            httpMethod_s,
            url_s,
            httpStatus_d,
            backendResponseCode_d,
            timeTaken_d,
            ResultType,
            apimSubscriptionId_s,
            clientIP_s
        | order by TimeGenerated desc
        | take 20
        """
        
        response = logs_client.query_workspace(
            workspace_id=LOG_ANALYTICS_WORKSPACE_ID,
            query=apim_query,
            timespan=timedelta(minutes=60)
        )
        
        if response.tables and len(response.tables[0].rows) > 0:
            table = response.tables[0]
            print(f"\n✅ Found {len(table.rows)} MCP requests through APIM:\n")
            print("-" * 120)
            print(f"{'Time':<12} {'Method':<6} {'Status':<7} {'Backend':<8} {'Duration':<10} {'Result':<12} {'Subscription':<22}")
            print("-" * 120)
            
            for row in table.rows[:15]:
                time_str = str(row[0])[11:19] if row[0] else "N/A"
                method = str(row[1])[:6] if row[1] else "N/A"
                status = str(int(row[3])) if row[3] else "N/A"
                backend = str(int(row[4])) if row[4] else "-"
                duration = f"{row[5]:.0f}ms" if row[5] else "N/A"
                result = str(row[6])[:12] if row[6] else "N/A"
                sub_id = str(row[7])[:22] if row[7] else "N/A"
                
                # Status icon
                status_icon = "✅" if result == "Succeeded" else "❌"
                
                print(f"{time_str:<12} {method:<6} {status:<7} {backend:<8} {duration:<10} {status_icon} {result:<10} {sub_id:<22}")
            print("-" * 120)
        else:
            print("\nℹ️  No MCP requests found in the last 30 minutes")
            print("   Try running some tests from Tutorial 18 first, then wait 2-5 minutes")
            
    except Exception as e:
        print(f"\n❌ Query failed: {e}")

## Part 4: Authentication Success & Failures

### 🏗️ CISO's Key Question

> *"Show me all failed authentication attempts - this is critical for security audits."*

This query filters for 401/403 status codes to identify:
- Invalid subscription keys
- Missing JWT tokens (after Tutorial 19)
- Expired credentials

In [ ]:
# Query 2: Authentication & Status Code Analysis
print("=" * 70)
print("🔐 Authentication Analysis")
print("=" * 70)

if not LOG_ANALYTICS_WORKSPACE_ID:
    print("\n⚠️  LOG_ANALYTICS_WORKSPACE_ID not configured - skipping query")
else:
    try:
        # First, let's see what status codes we actually have
        status_query = """
        AzureDiagnostics
        | where ResourceProvider == "MICROSOFT.APIMANAGEMENT"
        | where Category == "GatewayLogs"
        | where TimeGenerated > ago(1h)
        | where url_s contains "mcp"
        | summarize Count = count() by tostring(httpStatus_d), ResultType
        | order by Count desc
        """
        
        response = logs_client.query_workspace(
            workspace_id=LOG_ANALYTICS_WORKSPACE_ID,
            query=status_query,
            timespan=timedelta(hours=2)
        )
        
        if response.tables and len(response.tables[0].rows) > 0:
            print(f"\n📊 MCP API Status Code Distribution (Last Hour)\n")
            print("-" * 50)
            print(f"{'HTTP Status':<15} {'Result':<20} {'Count':<10}")
            print("-" * 50)
            
            total = 0
            succeeded = 0
            for row in response.tables[0].rows:
                status = str(row[0]) if row[0] else "N/A"
                result = str(row[1]) if row[1] else "N/A"
                count = int(row[2]) if row[2] else 0
                total += count
                
                # Track successes
                if result == "Succeeded":
                    succeeded += count
                
                # Status icon
                icon = "✅" if result == "Succeeded" else "❌"
                print(f"{status:<15} {icon} {result:<18} {count:<10}")
            
            print("-" * 50)
            print(f"{'TOTAL':<15} {'':<20} {total:<10}")
            
            if total > 0:
                success_rate = (succeeded / total) * 100
                print(f"\n📈 Success Rate: {success_rate:.1f}%")
                
                # Diagnostic hints based on status codes
                status_codes = {str(row[0]): int(row[2]) for row in response.tables[0].rows}
                
                if '404' in status_codes or '404.0' in status_codes:
                    count_404 = status_codes.get('404', 0) + status_codes.get('404.0', 0)
                    print(f"\n🔍 {count_404} Not Found (404) errors detected!")
                    print("   Possible causes:")
                    print("   • MCP server path incorrect (should be /mcp/mcp)")
                    print("   • API not registered in APIM")
                    print("   • Backend MCP server not running")
                
                if '401' in status_codes or '401.0' in status_codes:
                    count_401 = status_codes.get('401', 0) + status_codes.get('401.0', 0)
                    print(f"\n🔐 {count_401} Unauthorized (401) errors detected!")
                    print("   Possible causes: Invalid/missing subscription key or JWT token")
                    
                if '403' in status_codes or '403.0' in status_codes:
                    count_403 = status_codes.get('403', 0) + status_codes.get('403.0', 0)
                    print(f"\n🚫 {count_403} Forbidden (403) errors detected!")
                    print("   Possible causes: JWT token valid but lacks required claims/roles")
                    
                if '500' in status_codes or '502' in status_codes or '503' in status_codes:
                    print(f"\n💥 Server errors (5xx) detected!")
                    print("   Possible causes: Backend MCP server error or timeout")
        else:
            print("\nℹ️  No MCP requests found in the last hour")
            
    except Exception as e:
        print(f"\n❌ Query failed: {e}")

## Part 4b: JWT Token Claims Analysis (Debugging)

### 🏗️ Developer's Question

> *"What identity information is actually in the JWT token? I need to debug why my token is being rejected."*

This section decodes your current JWT token to show exactly what APIM's `validate-azure-ad-token` policy sees.

In [ ]:
# Decode and display JWT token claims being sent to APIM
# This shows exactly what identity information is in the token

import base64
import json
from datetime import datetime

def decode_jwt_payload(token: str) -> dict:
    """Decode a JWT token and return its claims (without verification)."""
    try:
        # JWT has 3 parts: header.payload.signature
        parts = token.split('.')
        if len(parts) != 3:
            return {"error": "Invalid JWT format"}
        
        # Decode the payload (middle part)
        payload = parts[1]
        # Add padding if needed
        padding = 4 - len(payload) % 4
        if padding != 4:
            payload += '=' * padding
        
        decoded = base64.urlsafe_b64decode(payload)
        return json.loads(decoded)
    except Exception as e:
        return {"error": str(e)}

print("=" * 70)
print("🔐 JWT Token Claims Analysis")
print("=" * 70)

# Try to get a token using DefaultAzureCredential
try:
    from azure.identity import DefaultAzureCredential
    
    # Get token for APIM (using the common Azure management scope)
    # You may need to adjust the scope based on your APIM configuration
    credential = DefaultAzureCredential()
    
    # Try to get a token - use your APIM's audience if configured
    # Common scopes: "https://management.azure.com/.default" or your custom API scope
    apim_scope = os.getenv("APIM_JWT_AUDIENCE", "https://management.azure.com/.default")
    
    print(f"\n📡 Requesting token for scope: {apim_scope}")
    token_response = credential.get_token(apim_scope)
    entra_token = token_response.token
    
    claims = decode_jwt_payload(entra_token)
    
    if "error" not in claims:
        print(f"""
✅ Token acquired successfully!

   Application (Client):
     • App ID (azp/appid): {claims.get('azp', claims.get('appid', 'N/A'))}
     • App Name: {claims.get('app_displayname', 'N/A')}
   
   Issuer & Audience:
     • Issuer (iss): {claims.get('iss', 'N/A')}
     • Audience (aud): {claims.get('aud', 'N/A')}
     • Tenant ID (tid): {claims.get('tid', 'N/A')}
   
   Token Validity:
     • Issued At: {datetime.fromtimestamp(claims.get('iat', 0)).strftime('%Y-%m-%d %H:%M:%S') if claims.get('iat') else 'N/A'}
     • Expires:   {datetime.fromtimestamp(claims.get('exp', 0)).strftime('%Y-%m-%d %H:%M:%S') if claims.get('exp') else 'N/A'}
   
   Identity:
     • Object ID (oid): {claims.get('oid', 'N/A')}
     • UPN/Name: {claims.get('upn', claims.get('unique_name', claims.get('name', 'N/A')))}
   
   Roles & Scopes:
     • Roles: {claims.get('roles', ['None'])}
     • Scopes (scp): {claims.get('scp', 'N/A')}
""")
        
        print("📋 What APIM's validate-azure-ad-token policy checks:")
        print("   ✓ Token is from the correct tenant (tid)")
        print("   ✓ Token is for the expected audience (aud)")  
        print("   ✓ Token is from an allowed client app (azp/appid)")
        print("   ✓ Token hasn't expired (exp)")
        
        # Show full claims for debugging
        print("\n📄 Full Token Claims:")
        print("-" * 50)
        for key, value in sorted(claims.items()):
            if key not in ['iat', 'exp', 'nbf']:  # Skip timestamps (already shown)
                val_str = str(value)[:60] + "..." if len(str(value)) > 60 else str(value)
                print(f"   {key}: {val_str}")
    else:
        print(f"\n❌ Could not decode token: {claims['error']}")
        
except Exception as e:
    print(f"\n⚠️  Could not acquire token: {e}")
    print("\n💡 This is expected if:")
    print("   • You're not logged in (run 'az login' first)")
    print("   • The scope doesn't match your APIM configuration")
    print("   • Set APIM_JWT_AUDIENCE in .env to your API's scope")

## Part 5: Latency Analysis for SLA Compliance

### 🏗️ Operations Team's Question

> *"Our SLA promises 95% of requests complete in under 500ms. Are we meeting it?"*

In [ ]:
# Query 3: Latency Percentiles for SLA
print("=" * 70)
print("⏱️  Latency Analysis (SLA Compliance)")
print("=" * 70)

if not LOG_ANALYTICS_WORKSPACE_ID:
    print("\n⚠️  LOG_ANALYTICS_WORKSPACE_ID not configured - skipping query")
else:
    try:
        latency_query = """
        AzureDiagnostics
        | where ResourceProvider == "MICROSOFT.APIMANAGEMENT"
        | where Category == "GatewayLogs"
        | where TimeGenerated > ago(1h)
        | where url_s contains "mcp"
        | where httpStatus_d == 200
        | summarize 
            RequestCount = count(),
            AvgLatency = avg(timeTaken_d),
            P50 = percentile(timeTaken_d, 50),
            P95 = percentile(timeTaken_d, 95),
            P99 = percentile(timeTaken_d, 99),
            MaxLatency = max(timeTaken_d)
        """
        
        response = logs_client.query_workspace(
            workspace_id=LOG_ANALYTICS_WORKSPACE_ID,
            query=latency_query,
            timespan=timedelta(hours=2)
        )
        
        if response.tables and len(response.tables[0].rows) > 0:
            row = response.tables[0].rows[0]
            count = int(row[0]) if row[0] else 0
            
            if count > 0:
                avg_lat = row[1] if row[1] else 0
                p50 = row[2] if row[2] else 0
                p95 = row[3] if row[3] else 0
                p99 = row[4] if row[4] else 0
                max_lat = row[5] if row[5] else 0
                
                print(f"\n📊 Latency Statistics (Last Hour, {count} successful requests)\n")
                print(f"   Average:     {avg_lat:.0f}ms")
                print(f"   P50 (Median):{p50:.0f}ms")
                print(f"   P95:         {p95:.0f}ms")
                print(f"   P99:         {p99:.0f}ms")
                print(f"   Maximum:     {max_lat:.0f}ms")
                
                # SLA check (assuming 500ms target at P95)
                sla_target = 500
                if p95 <= sla_target:
                    print(f"\n   ✅ SLA Met: P95 ({p95:.0f}ms) is under {sla_target}ms target")
                else:
                    print(f"\n   ❌ SLA Breach: P95 ({p95:.0f}ms) exceeds {sla_target}ms target")
            else:
                print("\nℹ️  No successful MCP requests found for latency analysis")
        else:
            print("\nℹ️  No data available")
            
    except Exception as e:
        print(f"\n❌ Query failed: {e}")

## Part 6: Tool Usage Breakdown

### 🏗️ Product Team's Question

> *"Which MCP tools are most popular? This helps us prioritize improvements."*

In [ ]:
# Query 4: Request Timeline (hourly breakdown)
print("=" * 70)
print("📈 Request Timeline (Last 6 Hours)")
print("=" * 70)

if not LOG_ANALYTICS_WORKSPACE_ID:
    print("\n⚠️  LOG_ANALYTICS_WORKSPACE_ID not configured - skipping query")
else:
    try:
        timeline_query = """
        AzureDiagnostics
        | where ResourceProvider == "MICROSOFT.APIMANAGEMENT"
        | where Category == "GatewayLogs"
        | where TimeGenerated > ago(6h)
        | where url_s contains "mcp"
        | summarize 
            Requests = count(),
            Succeeded = countif(httpStatus_d == 200),
            Failed = countif(httpStatus_d != 200)
          by bin(TimeGenerated, 1h)
        | order by TimeGenerated asc
        """
        
        response = logs_client.query_workspace(
            workspace_id=LOG_ANALYTICS_WORKSPACE_ID,
            query=timeline_query,
            timespan=timedelta(hours=8)
        )
        
        if response.tables and len(response.tables[0].rows) > 0:
            print(f"\n📊 Hourly Request Breakdown:\n")
            print("-" * 60)
            print(f"{'Hour':<20} {'Total':<10} {'✅ OK':<10} {'❌ Failed':<10}")
            print("-" * 60)
            
            for row in response.tables[0].rows:
                hour = str(row[0])[11:16] if row[0] else "N/A"
                total = int(row[1]) if row[1] else 0
                ok = int(row[2]) if row[2] else 0
                failed = int(row[3]) if row[3] else 0
                
                # Simple bar visualization
                bar = "█" * min(total, 20)
                print(f"{hour:<20} {total:<10} {ok:<10} {failed:<10} {bar}")
            print("-" * 60)
        else:
            print("\nℹ️  No MCP requests in the last 6 hours")
            
    except Exception as e:
        print(f"\n❌ Query failed: {e}")

## Part 7: Subscription Key Usage

### 🏗️ Security Team's Question

> *"Which subscription keys are being used, and from where? We need this for key rotation planning."*

In [ ]:
# Query 5: Subscription Key Usage
print("=" * 70)
print("🔑 Subscription Key Usage Analysis")
print("=" * 70)

if not LOG_ANALYTICS_WORKSPACE_ID:
    print("\n⚠️  LOG_ANALYTICS_WORKSPACE_ID not configured - skipping query")
else:
    try:
        subscription_query = """
        AzureDiagnostics
        | where ResourceProvider == "MICROSOFT.APIMANAGEMENT"
        | where Category == "GatewayLogs"
        | where TimeGenerated > ago(24h)
        | where url_s contains "mcp"
        | summarize 
            RequestCount = count(),
            SuccessCount = countif(httpStatus_d == 200),
            FailCount = countif(httpStatus_d != 200),
            UniqueIPs = dcount(clientIP_s),
            LastSeen = max(TimeGenerated)
          by apimSubscriptionId_s
        | order by RequestCount desc
        """
        
        response = logs_client.query_workspace(
            workspace_id=LOG_ANALYTICS_WORKSPACE_ID,
            query=subscription_query,
            timespan=timedelta(hours=48)
        )
        
        if response.tables and len(response.tables[0].rows) > 0:
            print(f"\n📊 Subscription Key Activity (Last 24 Hours):\n")
            print("-" * 90)
            print(f"{'Subscription ID':<30} {'Requests':<10} {'✅ OK':<8} {'❌ Fail':<8} {'IPs':<6} {'Last Seen':<15}")
            print("-" * 90)
            
            for row in response.tables[0].rows:
                sub_id = str(row[0])[:30] if row[0] else "(none)"
                requests = int(row[1]) if row[1] else 0
                success = int(row[2]) if row[2] else 0
                fail = int(row[3]) if row[3] else 0
                ips = int(row[4]) if row[4] else 0
                last_seen = str(row[5])[11:19] if row[5] else "N/A"
                
                print(f"{sub_id:<30} {requests:<10} {success:<8} {fail:<8} {ips:<6} {last_seen:<15}")
            print("-" * 90)
            
            print("\n💡 Recommendations:")
            print("   • Rotate keys that haven't been used in 30+ days")
            print("   • Investigate keys with high failure rates")
            print("   • Monitor keys used from multiple IPs (possible sharing)")
        else:
            print("\nℹ️  No subscription key activity found")
            
    except Exception as e:
        print(f"\n❌ Query failed: {e}")

## Part 8: Distributed Tracing - End-to-End Request Flow

### 🏗️ Architect's Key Question

> *"How do I trace a single request from JWT validation in APIM, through to the Container App, and into the MCP server?"*

### The Challenge: Connecting the Dots

```
┌─────────────┐     ┌─────────────┐     ┌─────────────────────────┐
│   Client    │ ──► │    APIM     │ ──► │   Container App (MCP)   │
│  (Agent)    │     │ JWT + Logs  │     │   Application Logs      │
└─────────────┘     └─────────────┘     └─────────────────────────┘
      │                   │                        │
      └───────────────────┴────────────────────────┘
                    Correlation ID links them all!
```

### Best Practice: Use Correlation Headers

APIM automatically generates and propagates these headers:

| Header | Purpose | Example |
|--------|---------|---------|
| `x-ms-request-id` | APIM's unique request ID | `abc123-def456` |
| `traceparent` | W3C Trace Context (if App Insights enabled) | `00-0af7651916cd43dd8448eb211c80319c-...` |
| `Request-Id` | Application Insights correlation | `\|abc123.def456` |

### How to Enable End-to-End Tracing

1. **APIM**: Enable Application Insights integration
2. **Container App**: Enable Application Insights or use OpenTelemetry
3. **MCP Server**: Log the correlation headers

### Architecture Options

| Option | Pros | Cons |
|--------|------|------|
| **App Insights (Recommended)** | Auto-correlation, built-in visualization | Requires App Insights on all services |
| **Custom Correlation ID** | Works everywhere, simple | Manual implementation required |
| **OpenTelemetry** | Vendor-neutral, flexible | More setup required |

In [ ]:
# Query: APIM Requests with Correlation IDs for Distributed Tracing
print("=" * 70)
print("🔗 Distributed Tracing - End-to-End Request Flow")
print("=" * 70)

if not LOG_ANALYTICS_WORKSPACE_ID:
    print("\n⚠️  LOG_ANALYTICS_WORKSPACE_ID not configured - skipping query")
else:
    try:
        # Query APIM Gateway Logs with correlation columns
        # Schema discovery found: CorrelationId, operationId_s, id_s, apiId_s
        trace_query = """
        AzureDiagnostics
        | where ResourceProvider == "MICROSOFT.APIMANAGEMENT"
        | where Category == "GatewayLogs"
        | where TimeGenerated > ago(1h)
        | where requestUri_s contains "mcp" or apiId_s contains "mcp"
        | project 
            TimeGenerated,
            CorrelationId,
            operationId_s,
            apiId_s,
            requestUri_s,
            isRequestSuccess_b,
            backendResponseCode_d,
            requestSize_d,
            apimSubscriptionId_s
        | order by TimeGenerated desc
        | take 15
        """
        
        response = logs_client.query_workspace(
            workspace_id=LOG_ANALYTICS_WORKSPACE_ID,
            query=trace_query,
            timespan=timedelta(hours=2)
        )
        
        if response.tables and len(response.tables[0].rows) > 0:
            table = response.tables[0]
            print(f"\n✅ Found {len(table.rows)} traced requests:\n")
            print("-" * 110)
            print(f"{'Time':<12} {'Correlation ID':<38} {'Success':<8} {'Backend':<8} {'API ID':<20}")
            print("-" * 110)
            
            for row in table.rows[:15]:
                time_str = str(row[0])[11:19] if row[0] else "N/A"
                corr_id = str(row[1])[:36] if row[1] else "N/A"
                success = "✅" if row[5] else "❌" if row[5] is not None else "?"
                backend = str(int(row[6])) if row[6] else "-"
                api_id = str(row[3])[:20] if row[3] else "N/A"
                
                print(f"{time_str:<12} {corr_id:<38} {success:<8} {backend:<8} {api_id:<20}")
            
            print("-" * 110)
            
            # Show a sample correlation ID for tracing
            sample_corr_id = str(table.rows[0][1]) if table.rows[0][1] else None
            if sample_corr_id:
                print(f"\n💡 Sample Correlation ID for tracing: {sample_corr_id}")
                print("   Use this ID to search in Container App logs and Application Insights")
        else:
            print("\nℹ️  No MCP requests found in the last hour")
            print("   Try running some tests from Tutorial 18 first, then wait 2-5 minutes")
            
    except Exception as e:
        print(f"\n❌ Query failed: {e}")

### Querying Container App Logs with Correlation ID

If your Container App sends logs to the same Log Analytics workspace, you can join them:

In [ ]:
# Query: Container App logs (may be in a different workspace)
print("=" * 70)
print("🔍 End-to-End Trace: APIM → Container App")
print("=" * 70)

# Container App logs might be in a different Log Analytics workspace
CONTAINER_APP_WORKSPACE_ID = os.getenv("CONTAINER_APP_LOG_ANALYTICS_WORKSPACE_ID", "")

# Use Container App workspace if specified, otherwise fall back to APIM workspace
workspace_to_query = CONTAINER_APP_WORKSPACE_ID or LOG_ANALYTICS_WORKSPACE_ID

if not workspace_to_query:
    print("\n⚠️  No Log Analytics workspace configured - skipping query")
    print("   Set CONTAINER_APP_LOG_ANALYTICS_WORKSPACE_ID in .env for Container App logs")
else:
    workspace_name = "Container App workspace" if CONTAINER_APP_WORKSPACE_ID else "APIM workspace"
    print(f"\n📡 Querying {workspace_name}: {workspace_to_query[:20]}...")
    
    try:
        # Query Container App Console Logs with correct schema from ContainerAppConsoleLogs_CL
        container_query = """
        ContainerAppConsoleLogs_CL
        | where TimeGenerated > ago(1h)
        | where ContainerAppName_s == "travel-mcp-server"
        | project 
            TimeGenerated,
            ContainerAppName_s,
            ContainerName_s,
            Log_s,
            Stream_s,
            ContainerId_g
        | order by TimeGenerated desc
        | take 15
        """
        
        response = logs_client.query_workspace(
            workspace_id=workspace_to_query,
            query=container_query,
            timespan=timedelta(hours=2)
        )
        
        if response.tables and len(response.tables[0].rows) > 0:
            table = response.tables[0]
            print(f"\n✅ Found {len(table.rows)} Container App log entries:\n")
            print("-" * 100)
            print(f"{'Time':<12} {'App':<20} {'Stream':<8} {'Log Message':<55}")
            print("-" * 100)
            
            for row in table.rows[:15]:
                time_str = str(row[0])[11:19] if row[0] else "N/A"
                app_name = str(row[1])[:18] if row[1] else "N/A"
                stream = str(row[4])[:8] if row[4] else "N/A"
                log_msg = str(row[3])[:55] if row[3] else "N/A"
                
                # Color-code based on stream (stdout vs stderr)
                stream_icon = "📤" if stream == "stdout" else "⚠️"
                print(f"{time_str:<12} {app_name:<20} {stream_icon} {stream:<6} {log_msg}")
            
            print("-" * 100)
            
            # Show container ID for correlation
            if table.rows[0][5]:
                print(f"\n🔗 Container ID: {table.rows[0][5]}")
                print("   Use this to correlate with APIM requests by time")
        else:
            print("\nℹ️  No Container App logs found for 'travel-mcp-server'")
            print("   Check that the Container App is running and logging")
            
    except Exception as e:
        error_msg = str(e)
        if "SemanticError" in error_msg or "BadArgumentError" in error_msg:
            print(f"\n⚠️  Table not found in this workspace")
            print("   The ContainerAppConsoleLogs_CL table may not be configured")
            print("\n💡 To enable Container App logs:")
            print("   1. Go to Container App → Monitoring → Diagnostic settings")
            print("   2. Add diagnostic setting → Send to Log Analytics workspace")
            print("   3. Select 'ContainerAppConsoleLogs'")
        else:
            print(f"\n❌ Query failed: {e}")

print(f"""
💡 Configuration Note:
   If Container App logs are in a different workspace, add to your .env:
   CONTAINER_APP_LOG_ANALYTICS_WORKSPACE_ID=<workspace-id>
   
   Current workspaces:
   • APIM:          {LOG_ANALYTICS_WORKSPACE_ID[:20] + '...' if LOG_ANALYTICS_WORKSPACE_ID else 'Not set'}
   • Container App: {CONTAINER_APP_WORKSPACE_ID[:20] + '...' if CONTAINER_APP_WORKSPACE_ID else 'Using APIM workspace'}
""")

### Best Practice: Enable Full Distributed Tracing

#### Option 1: Application Insights (Recommended)

**Step 1: Enable App Insights on APIM**
```
Azure Portal → APIM → Settings → Application Insights → Add
```

**Step 2: Enable App Insights on Container App**
```bash
az containerapp update \
  --name travel-mcp-server \
  --resource-group $RESOURCE_GROUP \
  --set-env-vars \
    APPLICATIONINSIGHTS_CONNECTION_STRING="InstrumentationKey=xxx;..."
```

**Step 3: Add OpenTelemetry to your MCP Server**
```python
# In travel_mcp_server.py
from opentelemetry import trace
from opentelemetry.instrumentation.fastapi import FastAPIInstrumentor
from azure.monitor.opentelemetry import configure_azure_monitor

# Configure Azure Monitor (App Insights)
configure_azure_monitor(
    connection_string=os.getenv("APPLICATIONINSIGHTS_CONNECTION_STRING")
)

# Auto-instrument FastAPI
FastAPIInstrumentor.instrument_app(app)
```

#### Option 2: Custom Correlation Header

If you can't use App Insights, pass a custom header through the chain:

**APIM Policy (inbound):**
```xml
<set-header name="X-Correlation-ID" exists-action="skip">
    <value>@(context.RequestId.ToString())</value>
</set-header>
```

**MCP Server (Python):**
```python
@app.middleware("http")
async def log_correlation(request: Request, call_next):
    correlation_id = request.headers.get("X-Correlation-ID", str(uuid.uuid4()))
    logger.info(f"[{correlation_id}] {request.method} {request.url.path}")
    response = await call_next(request)
    response.headers["X-Correlation-ID"] = correlation_id
    return response
```

In [ ]:
# Query: Correlated End-to-End Trace - APIM + Container App by Timestamp
import pandas as pd
from IPython.display import display, Markdown

# Get both workspace IDs
CONTAINER_APP_WORKSPACE_ID = os.getenv("CONTAINER_APP_LOG_ANALYTICS_WORKSPACE_ID", "")

if not LOG_ANALYTICS_WORKSPACE_ID or not CONTAINER_APP_WORKSPACE_ID:
    print("\n⚠️  Both workspace IDs required for correlation")
    print(f"   APIM Workspace:          {'✅' if LOG_ANALYTICS_WORKSPACE_ID else '❌ Not set'}")
    print(f"   Container App Workspace: {'✅' if CONTAINER_APP_WORKSPACE_ID else '❌ Not set'}")
else:
    try:
        # Query APIM logs
        apim_trace_query = """
        AzureDiagnostics
        | where ResourceProvider == "MICROSOFT.APIMANAGEMENT"
        | where Category == "GatewayLogs"
        | where TimeGenerated > ago(1h)
        | where requestUri_s contains "mcp" or apiId_s contains "mcp"
        | project APIMTime = TimeGenerated, CorrelationId, apiId_s, isRequestSuccess_b, backendResponseCode_d
        | order by APIMTime desc
        | take 10
        """
        
        apim_response = logs_client.query_workspace(
            workspace_id=LOG_ANALYTICS_WORKSPACE_ID,
            query=apim_trace_query,
            timespan=timedelta(hours=2)
        )
        
        # Query Container App logs
        container_trace_query = """
        ContainerAppConsoleLogs_CL
        | where TimeGenerated > ago(1h)
        | where ContainerAppName_s == "travel-mcp-server"
        | where Log_s contains "POST /mcp" or Log_s contains "mcp"
        | project ContainerTime = TimeGenerated, Log_s, ContainerId_g
        | order by ContainerTime desc
        | take 10
        """
        
        container_response = logs_client.query_workspace(
            workspace_id=CONTAINER_APP_WORKSPACE_ID,
            query=container_trace_query,
            timespan=timedelta(hours=2)
        )
        
        apim_rows = apim_response.tables[0].rows if apim_response.tables and apim_response.tables[0].rows else []
        container_rows = container_response.tables[0].rows if container_response.tables and container_response.tables[0].rows else []
        
        # Display header
        display(Markdown("## 🔗 End-to-End Request Correlation"))
        display(Markdown("*APIM Gateway → Container App (MCP Server)*"))
        
        # APIM DataFrame
        display(Markdown("### 📡 APIM Gateway (Entry Point)"))
        if apim_rows:
            apim_data = []
            for row in apim_rows[:8]:
                time_str = str(row[0])[11:19] if row[0] else "N/A"
                success = row[3]
                status = "✅ Success" if success else "❌ Failed"
                backend = str(int(row[4])) if row[4] else "—"
                corr_id = str(row[1])[:36] if row[1] else "N/A"
                apim_data.append({
                    "Time (UTC)": time_str,
                    "Status": status,
                    "Backend Code": backend,
                    "Correlation ID": corr_id
                })
            apim_df = pd.DataFrame(apim_data)
            display(apim_df)
        else:
            print("ℹ️ No APIM requests found in the last hour")
        
        # Arrow indicator
        display(Markdown("### ⬇️"))
        
        # Container App DataFrame
        display(Markdown("### 🐳 Container App (MCP Server)"))
        if container_rows:
            container_data = []
            for row in container_rows[:8]:
                time_str = str(row[0])[11:19] if row[0] else "N/A"
                log_msg = str(row[1])[:80] if row[1] else "N/A"
                is_success = "200" in log_msg
                status = "✅" if is_success else "❌"
                container_data.append({
                    "Time (UTC)": time_str,
                    "Status": status,
                    "Log Message": log_msg
                })
            container_df = pd.DataFrame(container_data)
            display(container_df)
        else:
            print("ℹ️ No Container App logs found")
        
        # Summary stats
        display(Markdown("### 📊 Summary"))
        apim_success = sum(1 for r in apim_rows if r[3])
        summary_data = {
            "Metric": ["APIM Requests", "APIM Success", "Container App Logs"],
            "Count": [len(apim_rows), apim_success, len(container_rows)]
        }
        summary_df = pd.DataFrame(summary_data)
        display(summary_df)
        
        # Tip
        display(Markdown("""
> 💡 **Tracing Tip:** Match timestamps between APIM and Container App logs to trace a request.  
> The Correlation ID can be used in Azure Portal → Application Insights → Transaction Search.
"""))
            
    except Exception as e:
        print(f"\n❌ Query failed: {e}")

### 🛡️ Query: Policy Enforcement Details

This query shows how APIM policies (rate limiting, JWT validation) are being enforced. Look for:
- **Rate limiting blocks** - when clients exceed request limits
- **JWT validation failures** - when tokens are invalid or missing
- **Policy execution times** - performance impact of security policies

The query looks back **48 hours** to capture policy events that may not occur frequently.

In [ ]:
# Query: Policy Enforcement Details (Rate Limiting, JWT Validation)
import pandas as pd
from IPython.display import display, Markdown

if not LOG_ANALYTICS_WORKSPACE_ID:
    print("\n⚠️  LOG_ANALYTICS_WORKSPACE_ID not set - skipping query")
else:
    print("=" * 70)
    print("🛡️ APIM Policy Enforcement Analysis (Last 48 Hours)")
    print("=" * 70)
    
    try:
        # Query for policy enforcement events - rate limiting and JWT validation
        policy_query = """
        AzureDiagnostics
        | where ResourceProvider == "MICROSOFT.APIMANAGEMENT"
        | where Category == "GatewayLogs"
        | where TimeGenerated > ago(48h)
        | extend PolicyResult = case(
            responseCode_d == 429, "🚫 Rate Limited",
            responseCode_d == 401, "🔑 JWT Invalid/Missing",
            responseCode_d == 403, "⛔ Forbidden",
            isRequestSuccess_b == true, "✅ Allowed",
            "❓ Other"
          )
        | extend PolicyType = case(
            responseCode_d == 429, "rate-limit",
            responseCode_d == 401, "validate-jwt",
            responseCode_d == 403, "authorization",
            "none"
          )
        | project 
            TimeGenerated,
            apiId_s,
            operationId_s,
            method_s,
            requestUri_s,
            responseCode_d,
            PolicyResult,
            PolicyType,
            apimSubscriptionId_s,
            clientIP_s,
            timeTaken_d,
            lastError_reason_s,
            lastError_message_s
        | order by TimeGenerated desc
        | take 50
        """
        
        response = logs_client.query_workspace(
            workspace_id=LOG_ANALYTICS_WORKSPACE_ID,
            query=policy_query,
            timespan=timedelta(hours=48)
        )
        
        if response.tables and len(response.tables[0].rows) > 0:
            table = response.tables[0]
            
            # Categorize results
            rate_limited = [r for r in table.rows if r[6] == "🚫 Rate Limited"]
            jwt_failures = [r for r in table.rows if r[6] == "🔑 JWT Invalid/Missing"]
            forbidden = [r for r in table.rows if r[6] == "⛔ Forbidden"]
            allowed = [r for r in table.rows if r[6] == "✅ Allowed"]
            
            # Summary statistics
            display(Markdown("### 📊 Policy Enforcement Summary (48 Hours)"))
            summary_data = {
                "Policy Type": ["✅ Allowed (Success)", "🚫 Rate Limited (429)", "🔑 JWT Invalid (401)", "⛔ Forbidden (403)"],
                "Count": [len(allowed), len(rate_limited), len(jwt_failures), len(forbidden)],
                "Description": [
                    "Requests that passed all policies",
                    "Blocked by rate-limit policy",
                    "Blocked by validate-jwt policy",
                    "Authorization denied"
                ]
            }
            summary_df = pd.DataFrame(summary_data)
            display(summary_df)
            
            # Show rate limiting details if any
            if rate_limited:
                display(Markdown("### 🚫 Rate Limiting Events"))
                rate_data = []
                for row in rate_limited[:10]:
                    time_str = str(row[0])[5:19] if row[0] else "N/A"
                    api_id = str(row[1])[:25] if row[1] else "N/A"
                    client_ip = str(row[9])[:15] if row[9] else "N/A"
                    sub_id = str(row[8])[:20] if row[8] else "N/A"
                    rate_data.append({
                        "Time": time_str,
                        "API": api_id,
                        "Client IP": client_ip,
                        "Subscription": sub_id
                    })
                rate_df = pd.DataFrame(rate_data)
                display(rate_df)
                print(f"\n💡 Rate limiting is working! {len(rate_limited)} requests were blocked.")
            
            # Show JWT validation failures if any
            if jwt_failures:
                display(Markdown("### 🔑 JWT Validation Failures"))
                jwt_data = []
                for row in jwt_failures[:10]:
                    time_str = str(row[0])[5:19] if row[0] else "N/A"
                    api_id = str(row[1])[:25] if row[1] else "N/A"
                    client_ip = str(row[9])[:15] if row[9] else "N/A"
                    error_reason = str(row[11])[:30] if row[11] else "N/A"
                    error_msg = str(row[12])[:40] if row[12] else "N/A"
                    jwt_data.append({
                        "Time": time_str,
                        "API": api_id,
                        "Client IP": client_ip,
                        "Error Reason": error_reason
                    })
                jwt_df = pd.DataFrame(jwt_data)
                display(jwt_df)
                print(f"\n🔒 JWT validation is working! {len(jwt_failures)} invalid tokens were rejected.")
            
            # Show forbidden requests if any
            if forbidden:
                display(Markdown("### ⛔ Authorization Failures (403)"))
                forbidden_data = []
                for row in forbidden[:10]:
                    time_str = str(row[0])[5:19] if row[0] else "N/A"
                    api_id = str(row[1])[:25] if row[1] else "N/A"
                    method = str(row[3]) if row[3] else "N/A"
                    uri = str(row[4])[:40] if row[4] else "N/A"
                    forbidden_data.append({
                        "Time": time_str,
                        "API": api_id,
                        "Method": method,
                        "URI": uri
                    })
                forbidden_df = pd.DataFrame(forbidden_data)
                display(forbidden_df)
            
            # Recent activity timeline
            display(Markdown("### ⏱️ Recent Policy Activity (All Types)"))
            timeline_data = []
            for row in table.rows[:15]:
                time_str = str(row[0])[5:19] if row[0] else "N/A"
                policy_result = str(row[6]) if row[6] else "N/A"
                status_code = int(row[5]) if row[5] else 0
                api_id = str(row[1])[:20] if row[1] else "N/A"
                latency = f"{row[10]:.0f}ms" if row[10] else "N/A"
                timeline_data.append({
                    "Time": time_str,
                    "Result": policy_result,
                    "Status": status_code,
                    "API": api_id,
                    "Latency": latency
                })
            timeline_df = pd.DataFrame(timeline_data)
            display(timeline_df)
            
            # Policy effectiveness summary
            total = len(table.rows)
            blocked = len(rate_limited) + len(jwt_failures) + len(forbidden)
            if total > 0:
                display(Markdown(f"""
### 📈 Policy Effectiveness

| Metric | Value |
|--------|-------|
| **Total Requests** | {total} |
| **Blocked by Policies** | {blocked} ({(blocked/total*100):.1f}%) |
| **Rate Limited** | {len(rate_limited)} |
| **JWT Rejected** | {len(jwt_failures)} |
| **Forbidden** | {len(forbidden)} |
| **Allowed Through** | {len(allowed)} ({(len(allowed)/total*100):.1f}%) |

> 💡 **Security Note:** A healthy API should show some blocked requests if rate limiting and JWT validation are active. Zero blocks might mean the policies aren't being tested.
"""))
            
        else:
            print("\nℹ️  No policy events found in the last 48 hours")
            print("   This could mean:")
            print("   • No requests have been made to the API")
            print("   • Diagnostic settings aren't configured for GatewayLogs")
            print("   • Try running tests from Tutorial 18/19 and wait 2-5 minutes")
            
    except Exception as e:
        print(f"\n❌ Query failed: {e}")
        print("\n💡 Troubleshooting:")
        print("   • Verify LOG_ANALYTICS_WORKSPACE_ID is correct")
        print("   • Ensure APIM has diagnostic settings enabled for GatewayLogs")
        print("   • Check that your Azure credentials have Log Analytics Reader permission")

## Summary: Complete Observability Stack

### 🎯 What the CISO Now Has

```
┌─────────────────────────────────────────────────────────────────────┐
│                    COMPLIANCE DASHBOARD                             │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   ✅ WHO accessed the MCP server?                                  │
│      → Subscription key + JWT claims tracking                      │
│                                                                     │
│   ✅ WHEN did they access it?                                      │
│      → Full timeline with hourly breakdown                         │
│                                                                     │
│   ✅ WHAT operations did they perform?                             │
│      → HTTP method and URL logging                                 │
│                                                                     │
│   ✅ Were there FAILED auth attempts?                              │
│      → 401/403 response tracking with diagnostics                  │
│                                                                     │
│   ✅ Are we meeting SLAs?                                          │
│      → P50/P95/P99 latency percentiles                            │
│                                                                     │
│   ✅ Can we trace a request end-to-end?                            │
│      → Correlation ID / Operation ID across services               │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### 🔗 Distributed Tracing Architecture

```
┌──────────┐      ┌──────────────────────────────┐      ┌─────────────────┐
│  Client  │      │           APIM               │      │  Container App  │
│  Agent   │─────►│  • JWT Validation            │─────►│  • MCP Server   │
│          │      │  • Subscription Key Check    │      │  • Travel Tools │
│          │      │  • Generate Correlation ID   │      │  • Log w/ ID    │
└──────────┘      └──────────────────────────────┘      └─────────────────┘
                            │                                    │
                            ▼                                    ▼
                  ┌──────────────────────────────────────────────────────┐
                  │              Log Analytics Workspace                  │
                  │  • AzureDiagnostics (APIM Gateway Logs)              │
                  │  • ContainerAppConsoleLogs                           │
                  │  • AppRequests / AppDependencies (if App Insights)   │
                  └──────────────────────────────────────────────────────┘
                            │
                            ▼
                  ┌──────────────────────────────────────────────────────┐
                  │         JOIN on Correlation ID / Operation ID        │
                  │              = End-to-End Request Trace              │
                  └──────────────────────────────────────────────────────┘
```

### 📋 Key Queries for Auditors

| Question | KQL Query | Table |
|----------|-----------|-------|
| All requests | `AzureDiagnostics \| where Category == "GatewayLogs"` | AzureDiagnostics |
| Auth failures | `... \| where httpStatus_d in (401, 403)` | AzureDiagnostics |
| Latency SLA | `... \| summarize percentile(timeTaken_d, 95)` | AzureDiagnostics |
| Key usage | `... \| summarize count() by apimSubscriptionId_s` | AzureDiagnostics |
| End-to-end trace | `AppRequests \| join AppDependencies on operation_Id` | App Insights |
| Container logs | `ContainerAppConsoleLogs \| where Log contains "correlation"` | Container App |

### 🏗️ Recommended Observability Setup

| Component | What to Enable | Benefit |
|-----------|---------------|---------|
| **APIM** | Diagnostic Settings → GatewayLogs | See all API requests |
| **APIM** | Application Insights integration | Distributed tracing |
| **Container App** | Diagnostic Settings → Console Logs | See MCP server logs |
| **Container App** | App Insights connection string | Auto-correlation |
| **MCP Server** | OpenTelemetry + Azure Monitor | Full trace visibility |

### 🔜 What's Next

| Tutorial | Topic |
|----------|-------|
| **19** | Add JWT validation for user-level tracking |
| **19c** | Complete security flow with agent integration |

### Key Documentation

- [APIM Diagnostic Logs](https://learn.microsoft.com/azure/api-management/api-management-howto-use-azure-monitor)
- [APIM + Application Insights](https://learn.microsoft.com/azure/api-management/api-management-howto-app-insights)
- [Container Apps Logging](https://learn.microsoft.com/azure/container-apps/log-streaming)
- [Azure Monitor OpenTelemetry](https://learn.microsoft.com/azure/azure-monitor/app/opentelemetry-enable)
- [Distributed Tracing](https://learn.microsoft.com/azure/azure-monitor/app/distributed-trace-data)

---

**Congratulations!** You now have a complete observability story - from JWT validation through APIM to your MCP server. The CISO, security team, and operations team are all happy! 📊🔗